# Oxford Nanopore Sequencing Run QC

**Input:** `sequencing_summary_*.txt` output from ONT MinKNOW basecalling  
**Purpose:** Assess run quality across read length, Q-score, throughput, and channel activity before proceeding to SV/CNV analysis

---

## 0. Configuration

In [ ]:
# Path to the sequencing summary file produced by MinKNOW / Guppy
SEQUENCING_SUMMARY_PATH = "sequencing_summary_FAY50208_0ab7b0aa_5fa1353a.txt"

# Minimum Q-score threshold used during basecalling (ONT default = 8 for DNA)
QSCORE_PASS_THRESHOLD = 8

# Minimum read length filter applied upstream (must match pipeline config)
MIN_READ_LENGTH = 1000

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

## 2. Load Sequencing Summary

In [ ]:
COLUMN_NAMES = [
    "filename_fastq", "filename_fast5", "filename_pod5",
    "parent_read_id", "read_id", "run_id", "channel", "mux",
    "minknow_events", "start_time", "duration", "passes_filtering",
    "template_start", "num_events_template", "template_duration",
    "sequence_length_template", "mean_qscore_template",
    "median_template", "mad_template", "pore_type",
    "experiment_id", "sample_id", "end_reason"
]

df = pd.read_csv(
    SEQUENCING_SUMMARY_PATH,
    delimiter="\t",
    header=None,
    names=COLUMN_NAMES
)

# Coerce numeric columns
df["mean_qscore_template"]      = pd.to_numeric(df["mean_qscore_template"],      errors="coerce")
df["sequence_length_template"]  = pd.to_numeric(df["sequence_length_template"],  errors="coerce")
df["duration"]                  = pd.to_numeric(df["duration"],                  errors="coerce")
df["start_time"]                = pd.to_numeric(df["start_time"],                errors="coerce")
df["channel"]                   = pd.to_numeric(df["channel"],                   errors="coerce")

print(f"Total reads loaded : {len(df):,}")
df.head(3)

## 3. Run-Level Summary Statistics

In [ ]:
passed = df[df["passes_filtering"] == True]
failed = df[df["passes_filtering"] == False]

total_bases   = df["sequence_length_template"].sum()
passed_bases  = passed["sequence_length_template"].sum()

summary = {
    "Total reads"            : f"{len(df):,}",
    "Passed reads"           : f"{len(passed):,}  ({100*len(passed)/len(df):.1f}%)",
    "Failed reads"           : f"{len(failed):,}  ({100*len(failed)/len(df):.1f}%)",
    "Total yield (Gb)"       : f"{total_bases / 1e9:.2f}",
    "Passed yield (Gb)"      : f"{passed_bases / 1e9:.2f}",
    "Median read length (bp)": f"{df['sequence_length_template'].median():,.0f}",
    "N50 read length (bp)"   : f"{df.sort_values('sequence_length_template', ascending=False)['sequence_length_template'].cumsum().searchsorted(total_bases / 2):,}",
    "Median Q-score"         : f"{df['mean_qscore_template'].median():.1f}",
    "Mean Q-score"           : f"{df['mean_qscore_template'].mean():.1f}",
}

for k, v in summary.items():
    print(f"  {k:<30} {v}")

## 4. Q-Score Distribution — All Reads vs Passed Reads

The vertical dashed line marks the pass/fail Q-score threshold used during basecalling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, data, title in zip(
    axes,
    [df, passed],
    ["All Reads", "Passed Reads Only"]
):
    ax.hist(data["mean_qscore_template"].dropna(), bins=40,
            edgecolor="white", linewidth=0.4, color="steelblue")
    ax.axvline(QSCORE_PASS_THRESHOLD, color="crimson",
               linestyle="--", linewidth=1.5,
               label=f"Pass threshold (Q{QSCORE_PASS_THRESHOLD})")
    ax.set_xlabel("Mean Q-Score")
    ax.set_ylabel("Read Count")
    ax.set_title(f"Q-Score Distribution — {title}")
    ax.set_xlim(0, 25)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("qscore_distribution.png", bbox_inches="tight")
plt.show()

## 5. Read Length Distribution

Log-scale x-axis better represents the wide span of ONT read lengths. The vertical line marks the minimum read length filter applied upstream.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title, color in zip(
    axes,
    [df, passed],
    ["All Reads", "Passed Reads Only"],
    ["steelblue", "seagreen"]
):
    lengths = data["sequence_length_template"].dropna()
    log_bins = np.logspace(np.log10(max(lengths.min(), 1)),
                           np.log10(lengths.max()), 60)
    ax.hist(lengths, bins=log_bins, edgecolor="white",
            linewidth=0.3, color=color)
    ax.axvline(MIN_READ_LENGTH, color="crimson", linestyle="--",
               linewidth=1.5, label=f"Min length filter ({MIN_READ_LENGTH} bp)")
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{int(x):,}"))
    ax.set_xlabel("Read Length (bp, log scale)")
    ax.set_ylabel("Read Count")
    ax.set_title(f"Read Length Distribution — {title}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("read_length_distribution.png", bbox_inches="tight")
plt.show()

## 6. Read Length vs Q-Score Scatter

Passed reads are overlaid in a different color. A negative correlation between read length and Q-score is common in ONT data — longer reads are harder to basecall accurately.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Sample for performance if dataset is large
plot_df = df.sample(min(50_000, len(df)), random_state=42)

ax.scatter(
    plot_df.loc[plot_df["passes_filtering"] == False, "sequence_length_template"],
    plot_df.loc[plot_df["passes_filtering"] == False, "mean_qscore_template"],
    s=1, alpha=0.3, color="lightcoral", label="Failed"
)
ax.scatter(
    plot_df.loc[plot_df["passes_filtering"] == True, "sequence_length_template"],
    plot_df.loc[plot_df["passes_filtering"] == True, "mean_qscore_template"],
    s=1, alpha=0.3, color="steelblue", label="Passed"
)

ax.axhline(QSCORE_PASS_THRESHOLD, color="crimson", linestyle="--",
           linewidth=1.2, label=f"Q{QSCORE_PASS_THRESHOLD} threshold")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xlabel("Read Length (bp, log scale)")
ax.set_ylabel("Mean Q-Score")
ax.set_title("Read Length vs Q-Score")
ax.legend(markerscale=6, fontsize=9)

plt.tight_layout()
plt.savefig("length_vs_qscore.png", bbox_inches="tight")
plt.show()

## 7. Sequencing Throughput Over Time

Cumulative bases generated over the course of the run. A healthy run shows a roughly linear or slightly decelerating curve as pores become occupied or inactive.

In [ ]:
throughput = (
    passed
    .dropna(subset=["start_time", "sequence_length_template"])
    .sort_values("start_time")
    .assign(cumulative_gb=lambda x:
            x["sequence_length_template"].cumsum() / 1e9)
    .assign(start_time_h=lambda x: x["start_time"] / 3600)
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(throughput["start_time_h"], throughput["cumulative_gb"],
        color="steelblue", linewidth=1.2)
ax.set_xlabel("Run Time (hours)")
ax.set_ylabel("Cumulative Yield (Gb)")
ax.set_title("Sequencing Throughput Over Time (Passed Reads)")

plt.tight_layout()
plt.savefig("throughput_over_time.png", bbox_inches="tight")
plt.show()

## 8. Active Channel / Pore Utilization Heatmap

The MinION flow cell has 512 channels arranged in a 32×16 grid. This heatmap shows read count per channel — dark channels indicate inactive or blocked pores.

In [ ]:
channel_counts = (
    passed
    .groupby("channel")["read_id"]
    .count()
    .reindex(range(1, 513), fill_value=0)
)

# Reshape into 32 x 16 grid (MinION layout)
grid = channel_counts.values.reshape(32, 16)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    grid,
    ax=ax,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Read Count"},
    xticklabels=False,
    yticklabels=False
)
ax.set_title("Pore Activity Heatmap — Read Count per Channel (Passed Reads)")
ax.set_xlabel("Channel Column")
ax.set_ylabel("Channel Row")

active = (channel_counts > 0).sum()
print(f"Active channels : {active} / 512  ({100*active/512:.1f}%)")

plt.tight_layout()
plt.savefig("pore_activity_heatmap.png", bbox_inches="tight")
plt.show()

## 9. Read End Reason Distribution

Shows why reads were terminated. `signal_positive` is normal. High rates of `mux_change` or `unblock_mux_change` can indicate pore-blocking issues.

In [ ]:
end_reason_counts = df["end_reason"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
end_reason_counts.plot(kind="bar", ax=ax, color="steelblue",
                       edgecolor="white", linewidth=0.5)
ax.set_xlabel("End Reason")
ax.set_ylabel("Read Count")
ax.set_title("Read End Reason Distribution")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("end_reason_distribution.png", bbox_inches="tight")
plt.show()

print(end_reason_counts.to_string())

## 10. Reads Surviving Upstream Length Filter

Shows what fraction of passed reads would be retained after the `filtlong --min_length` filter applied in the SV pipeline.

In [ ]:
surviving = passed[passed["sequence_length_template"] >= MIN_READ_LENGTH]
discarded = passed[passed["sequence_length_template"] < MIN_READ_LENGTH]

labels = [f"Retained\n(≥{MIN_READ_LENGTH:,} bp)", f"Discarded\n(<{MIN_READ_LENGTH:,} bp)"]
sizes  = [len(surviving), len(discarded)]
colors = ["steelblue", "lightcoral"]

fig, ax = plt.subplots(figsize=(5, 5))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct="%1.1f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
ax.set_title(f"Passed Reads Surviving Length Filter ({MIN_READ_LENGTH:,} bp)")

plt.tight_layout()
plt.savefig("length_filter_retention.png", bbox_inches="tight")
plt.show()

print(f"Retained : {len(surviving):,}  ({100*len(surviving)/len(passed):.1f}%)")
print(f"Discarded: {len(discarded):,}  ({100*len(discarded)/len(passed):.1f}%)")
print(f"Retained yield: {surviving['sequence_length_template'].sum()/1e9:.2f} Gb")

## 11. Export QC Summary Table

In [ ]:
qc_export = pd.DataFrame([
    {"Metric": k, "Value": v} for k, v in summary.items()
])
qc_export = pd.concat([
    qc_export,
    pd.DataFrame([
        {"Metric": f"Reads retained after {MIN_READ_LENGTH}bp filter",
         "Value": f"{len(surviving):,}  ({100*len(surviving)/len(passed):.1f}%)"},
        {"Metric": "Yield after length filter (Gb)",
         "Value": f"{surviving['sequence_length_template'].sum()/1e9:.2f}"},
        {"Metric": "Active channels",
         "Value": f"{active} / 512  ({100*active/512:.1f}%)"},
    ])
], ignore_index=True)

qc_export.to_csv("sequencing_qc_summary.csv", index=False)
print("QC summary saved to: sequencing_qc_summary.csv")
qc_export